# Évaluation et test du modèle

Ce notebook permet d'évaluer en profondeur le modèle entraîné et de le tester avec des exemples.


In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json

# Add parent directory to path
sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath(''))))

from src.data_preparation import prepare_diagnostic_data, DataPreparator
from src.utils import load_model, load_metadata, calculate_metrics, print_classification_report
from sklearn.metrics import confusion_matrix, classification_report

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)


## Chargement du modèle et des données


In [ ]:
# Load model and preprocessors
models_dir = "../experiments/best_models"
model_path = os.path.join(models_dir, "diagnostic_model.pkl")
scaler_path = os.path.join(models_dir, "scaler.pkl")
encoder_path = os.path.join(models_dir, "label_encoder.pkl")
feature_cols_path = os.path.join(models_dir, "feature_columns.pkl")

# Load model
model = load_model(model_path)
print(f"✓ Modèle chargé: {type(model).__name__}")

# Load preprocessors
scaler = joblib.load(scaler_path)
label_encoder = joblib.load(encoder_path)
feature_columns = joblib.load(feature_cols_path)

print(f"✓ Scaler chargé")
print(f"✓ Label encoder chargé: {len(label_encoder.classes_)} classes")
print(f"✓ Feature columns chargées: {len(feature_columns)} features")

# Load metadata
metadata = load_metadata(model_path)
if metadata:
    print(f"\nMétadonnées du modèle:")
    print(json.dumps(metadata, indent=2))


In [ ]:
# Prepare test data
data = prepare_diagnostic_data(
    data_path="../data/raw/symptoms_disease.csv",
    test_size=0.2
)

X_train = data["X_train"]
X_test = data["X_test"]
y_train = data["y_train"]
y_test = data["y_test"]

print(f"Train set: {X_train.shape}")
print(f"Test set: {X_test.shape}")
print(f"\nClasses dans le test set: {y_test.nunique()}")


## Évaluation sur le jeu de test


In [ ]:
# Scale test data
X_test_scaled = scaler.transform(X_test)

# Make predictions
y_pred = model.predict(X_test_scaled)
y_pred_proba = model.predict_proba(X_test_scaled) if hasattr(model, 'predict_proba') else None

# Calculate metrics
metrics = calculate_metrics(y_test.values, y_pred)
print("\n" + "="*60)
print("MÉTRIQUES SUR LE JEU DE TEST")
print("="*60)
for metric, value in metrics.items():
    print(f"{metric.upper()}: {value:.4f}")
print("="*60)


## Matrice de confusion


In [ ]:
# Create confusion matrix
cm = confusion_matrix(y_test.values, y_pred)
class_names = label_encoder.classes_

# Plot confusion matrix
plt.figure(figsize=(14, 12))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=class_names, yticklabels=class_names,
            cbar_kws={'label': 'Nombre de prédictions'})
plt.title('Matrice de confusion', fontsize=16, fontweight='bold')
plt.xlabel('Prédictions', fontsize=12)
plt.ylabel('Vraies valeurs', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

# Calculate accuracy per class
class_accuracy = cm.diagonal() / cm.sum(axis=1)
print("\nPrécision par classe:")
for i, (class_name, acc) in enumerate(zip(class_names, class_accuracy)):
    print(f"  {class_name}: {acc:.4f}")


## Rapport de classification détaillé


In [ ]:
# Print detailed classification report
print_classification_report(y_test.values, y_pred, class_names=class_names)


## Analyse des erreurs


In [ ]:
# Find misclassified examples
misclassified = y_test.values != y_pred
n_errors = misclassified.sum()

print(f"Nombre d'erreurs: {n_errors} sur {len(y_test)} ({n_errors/len(y_test)*100:.2f}%)")

if n_errors > 0:
    # Get error details
    error_indices = np.where(misclassified)[0]
    
    # Sample some errors for analysis
    n_samples = min(10, len(error_indices))
    sample_errors = np.random.choice(error_indices, n_samples, replace=False)
    
    print(f"\nExemples d'erreurs (échantillon de {n_samples}):")
    print("-" * 80)
    
    for idx in sample_errors:
        true_label = label_encoder.inverse_transform([y_test.iloc[idx]])[0]
        pred_label = label_encoder.inverse_transform([y_pred[idx]])[0]
        confidence = y_pred_proba[idx].max() if y_pred_proba is not None else None
        
        print(f"\nExemple {idx}:")
        print(f"  Vraie maladie: {true_label}")
        print(f"  Prédiction: {pred_label}")
        if confidence:
            print(f"  Confiance: {confidence:.4f}")
        
        # Show symptoms for this case
        symptoms = X_test.iloc[idx]
        active_symptoms = symptoms[symptoms > 0]
        if len(active_symptoms) > 0:
            symptom_names = [feature_columns[i] for i in active_symptoms.index if i < len(feature_columns)]
            print(f"  Symptômes actifs: {', '.join(symptom_names[:10])}")


## Test avec des exemples personnalisés


In [ ]:
def predict_disease_from_symptoms(symptom_indices, model, scaler, label_encoder, feature_columns):
    """
    Prédire une maladie à partir d'une liste d'indices de symptômes.
    
    Args:
        symptom_indices: Liste d'indices de symptômes (0 ou 1 pour chaque symptôme)
        model: Modèle entraîné
        scaler: Scaler pour normaliser les features
        label_encoder: Encoder pour les labels
        feature_columns: Liste des noms de colonnes de features
        
    Returns:
        Dictionnaire avec la prédiction et les probabilités
    """
    # Create feature vector
    if len(symptom_indices) == len(feature_columns):
        # Binary vector provided
        feature_vector = np.array(symptom_indices).reshape(1, -1)
    else:
        # Indices provided - create binary vector
        feature_vector = np.zeros(len(feature_columns))
        for idx in symptom_indices:
            if 0 <= idx < len(feature_columns):
                feature_vector[idx] = 1
        feature_vector = feature_vector.reshape(1, -1)
    
    # Scale
    feature_vector_scaled = scaler.transform(feature_vector)
    
    # Predict
    prediction = model.predict(feature_vector_scaled)[0]
    predicted_disease = label_encoder.inverse_transform([prediction])[0]
    
    # Get probabilities
    if hasattr(model, 'predict_proba'):
        probabilities = model.predict_proba(feature_vector_scaled)[0]
        prob_dict = {
            label_encoder.inverse_transform([i])[0]: float(prob)
            for i, prob in enumerate(probabilities)
        }
        confidence = float(probabilities.max())
    else:
        prob_dict = None
        confidence = 1.0
    
    # Get active symptom names
    active_symptoms = [feature_columns[i] for i, val in enumerate(feature_vector[0]) if val > 0]
    
    return {
        'predicted_disease': predicted_disease,
        'confidence': confidence,
        'probabilities': prob_dict,
        'active_symptoms': active_symptoms
    }

# Test avec un exemple du dataset
print("Test avec un exemple du jeu de test:")
print("-" * 60)
sample_idx = 0
sample_symptoms = X_test.iloc[sample_idx].values
sample_true_label = label_encoder.inverse_transform([y_test.iloc[sample_idx]])[0]

result = predict_disease_from_symptoms(sample_symptoms, model, scaler, label_encoder, feature_columns)

print(f"Vraie maladie: {sample_true_label}")
print(f"Prédiction: {result['predicted_disease']}")
print(f"Confiance: {result['confidence']:.4f}")
print(f"Correct: {sample_true_label == result['predicted_disease']}")
print(f"\nTop 3 probabilités:")
sorted_probs = sorted(result['probabilities'].items(), key=lambda x: x[1], reverse=True)[:3]
for disease, prob in sorted_probs:
    print(f"  {disease}: {prob:.4f}")


In [ ]:
if y_pred_proba is not None:
    # Get confidence scores
    confidences = y_pred_proba.max(axis=1)
    
    # Separate correct and incorrect predictions
    correct_conf = confidences[~misclassified]
    incorrect_conf = confidences[misclassified]
    
    # Plot distribution
    plt.figure(figsize=(12, 6))
    
    plt.subplot(1, 2, 1)
    plt.hist(confidences, bins=30, alpha=0.7, edgecolor='black')
    plt.xlabel('Confiance', fontsize=12)
    plt.ylabel('Fréquence', fontsize=12)
    plt.title('Distribution des confiances', fontsize=14, fontweight='bold')
    plt.grid(True, alpha=0.3)
    
    plt.subplot(1, 2, 2)
    if len(correct_conf) > 0:
        plt.hist(correct_conf, bins=30, alpha=0.7, label='Correctes', color='green', edgecolor='black')
    if len(incorrect_conf) > 0:
        plt.hist(incorrect_conf, bins=30, alpha=0.7, label='Incorrectes', color='red', edgecolor='black')
    plt.xlabel('Confiance', fontsize=12)
    plt.ylabel('Fréquence', fontsize=12)
    plt.title('Confiance: Correctes vs Incorrectes', fontsize=14, fontweight='bold')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print(f"\nStatistiques de confiance:")
    print(f"  Moyenne: {confidences.mean():.4f}")
    print(f"  Médiane: {np.median(confidences):.4f}")
    print(f"  Min: {confidences.min():.4f}")
    print(f"  Max: {confidences.max():.4f}")
    print(f"\n  Moyenne (prédictions correctes): {correct_conf.mean():.4f if len(correct_conf) > 0 else 'N/A'}")
    print(f"  Moyenne (prédictions incorrectes): {incorrect_conf.mean():.4f if len(incorrect_conf) > 0 else 'N/A'}")


## Résumé et prochaines étapes

Le modèle est maintenant évalué et prêt à être utilisé. Les prochaines étapes peuvent inclure:
1. **Tester l'API** - Démarrer le serveur FastAPI et tester les endpoints
2. **Créer une interface utilisateur** - Interface web ou application pour utiliser le modèle
3. **Améliorer le modèle** - Si nécessaire, ajuster les hyperparamètres ou essayer d'autres modèles
4. **Déployer** - Mettre en production le modèle via Docker ou un service cloud
